In [1]:
import numpy as np
import pandas as pd

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error, r2_score

In [2]:
diabetes = load_diabetes()

X = diabetes.data
y = diabetes.target

print("Dataset loaded successfully")
print("Feature shape:", X.shape)
print("Target shape:", y.shape)

Dataset loaded successfully
Feature shape: (442, 10)
Target shape: (442,)


In [3]:
df = pd.DataFrame(X, columns=diabetes.feature_names)
df["target"] = y

print(df.head())

        age       sex       bmi        bp        s1        s2        s3  \
0  0.038076  0.050680  0.061696  0.021872 -0.044223 -0.034821 -0.043401   
1 -0.001882 -0.044642 -0.051474 -0.026328 -0.008449 -0.019163  0.074412   
2  0.085299  0.050680  0.044451 -0.005670 -0.045599 -0.034194 -0.032356   
3 -0.089063 -0.044642 -0.011595 -0.036656  0.012191  0.024991 -0.036038   
4  0.005383 -0.044642 -0.036385  0.021872  0.003935  0.015596  0.008142   

         s4        s5        s6  target  
0 -0.002592  0.019907 -0.017646   151.0  
1 -0.039493 -0.068332 -0.092204    75.0  
2 -0.002592  0.002861 -0.025930   141.0  
3  0.034309  0.022688 -0.009362   206.0  
4 -0.002592 -0.031988 -0.046641   135.0  


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training data shape:", X_train.shape)
print("Testing data shape:", X_test.shape)

Training data shape: (353, 10)
Testing data shape: (89, 10)


In [5]:
linear_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LinearRegression())
])

linear_model.fit(X_train, y_train)

y_pred_linear = linear_model.predict(X_test)

mse_linear = mean_squared_error(y_test, y_pred_linear)
r2_linear = r2_score(y_test, y_pred_linear)

print("Linear Regression")
print("MSE:", mse_linear)
print("R-squared:", r2_linear)

Linear Regression
MSE: 2900.193628493482
R-squared: 0.4526027629719195


In [6]:
ridge_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Ridge())
])

ridge_params = {
    "model__alpha": [0.01, 0.1, 1, 10, 100]
}

ridge_grid = GridSearchCV(
    ridge_pipeline,
    ridge_params,
    cv=5,
    scoring="neg_mean_squared_error"
)

ridge_grid.fit(X_train, y_train)

print("Best Ridge Alpha:", ridge_grid.best_params_)

Best Ridge Alpha: {'model__alpha': 10}


In [7]:
best_ridge = ridge_grid.best_estimator_

y_pred_ridge = best_ridge.predict(X_test)

mse_ridge = mean_squared_error(y_test, y_pred_ridge)
r2_ridge = r2_score(y_test, y_pred_ridge)

print("Ridge Regression")
print("MSE:", mse_ridge)
print("R-squared:", r2_ridge)

Ridge Regression
MSE: 2875.7787184218428
R-squared: 0.4572109567780849


In [8]:
lasso_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Lasso(max_iter=10000))
])

lasso_params = {
    "model__alpha": [0.001, 0.01, 0.1, 1, 10]
}

lasso_grid = GridSearchCV(
    lasso_pipeline,
    lasso_params,
    cv=5,
    scoring="neg_mean_squared_error"
)

lasso_grid.fit(X_train, y_train)

print("Best Lasso Alpha:", lasso_grid.best_params_)

Best Lasso Alpha: {'model__alpha': 1}


In [9]:
best_lasso = lasso_grid.best_estimator_

y_pred_lasso = best_lasso.predict(X_test)

mse_lasso = mean_squared_error(y_test, y_pred_lasso)
r2_lasso = r2_score(y_test, y_pred_lasso)

print("Lasso Regression")
print("MSE:", mse_lasso)
print("R-squared:", r2_lasso)

Lasso Regression
MSE: 2824.568094049959
R-squared: 0.46687670944102466


In [10]:
results = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Ridge Regression",
        "Lasso Regression"
    ],
    "MSE": [
        mse_linear,
        mse_ridge,
        mse_lasso
    ],
    "R-squared": [
        r2_linear,
        r2_ridge,
        r2_lasso
    ]
})

print("MODEL COMPARISON")
print("==============================")
print(results.to_string(index=False))

MODEL COMPARISON
            Model         MSE  R-squared
Linear Regression 2900.193628   0.452603
 Ridge Regression 2875.778718   0.457211
 Lasso Regression 2824.568094   0.466877


In [11]:
print("Best Ridge Alpha:", ridge_grid.best_params_["model__alpha"])
print("Best Lasso Alpha:", lasso_grid.best_params_["model__alpha"])

Best Ridge Alpha: 10
Best Lasso Alpha: 1


In [12]:
best_model = results.loc[results["MSE"].idxmin()]

print("Best Model:")
print(best_model["Model"])

print("\nLowest MSE:", best_model["MSE"])
print("R-squared:", best_model["R-squared"])

Best Model:
Lasso Regression

Lowest MSE: 2824.568094049959
R-squared: 0.46687670944102466


In [13]:
comparison = pd.DataFrame({
    "Actual": y_test,
    "Linear Prediction": y_pred_linear,
    "Ridge Prediction": y_pred_ridge,
    "Lasso Prediction": y_pred_lasso
})

print(comparison.head(10))

   Actual  Linear Prediction  Ridge Prediction  Lasso Prediction
0   219.0         139.547558        141.155432        144.570230
1    70.0         179.517208        180.289014        178.811105
2   202.0         134.038756        139.869495        146.735675
3   230.0         291.417029        292.195947        280.889578
4   111.0         123.789659        122.628518        124.039097
5    84.0          92.172347         94.721638         97.502499
6   242.0         258.232389        255.911393        259.677900
7   272.0         181.337321        186.982539        194.718738
8    94.0          90.224113         85.917515         88.391904
9    96.0         108.633759        111.722456        118.011017
